In [1]:
import pandas as pd
import numpy as np

# Load cleaned dataset
df = pd.read_csv("../data/processed/construction_cost_estimation_cleaned.csv")

# Display first five rows
df.head()

,project_id,year,district,building_type,construction_area_sqft,num_floors,bedrooms,bathrooms,construction_quality,foundation_type,material_price_index,labor_rate_per_sqft,site_condition,project_duration_months,total_construction_cost_inr
0,TS-SYN-000001,2024,Medchal-Malkajgiri,Apartment,840,5,2,2,Premium,Pile,149.7,733,Sloped,11,4614000
1,TS-SYN-000002,2022,Rangareddy,Duplex,4380,2,8,7,Basic,Shallow,136.5,412,Normal,11,7711000
2,TS-SYN-000003,2025,Hyderabad,Independent House,1180,1,2,3,Standard,Shallow,149.8,698,Normal,5,3913000
3,TS-SYN-000004,2024,Nalgonda,Independent House,2100,1,5,4,Basic,Shallow,143.5,360,Normal,7,3574000
4,TS-SYN-000005,2018,Hyderabad,Independent House,1340,1,4,5,Standard,Shallow,106.1,453,Sloped,7,2983000


In [2]:
print("Dataset Shape:", df.shape)

print("\nDataset Columns:")
print(df.columns.tolist())

print("\nDataset Information:")
df.info()

Dataset Shape: (12000, 15)

Dataset Columns:
['project_id', 'year', 'district', 'building_type', 'construction_area_sqft', 'num_floors', 'bedrooms', 'bathrooms', 'construction_quality', 'foundation_type', 'material_price_index', 'labor_rate_per_sqft', 'site_condition', 'project_duration_months', 'total_construction_cost_inr']

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   project_id                   12000 non-null  str    
 1   year                         12000 non-null  int64  
 2   district                     12000 non-null  str    
 3   building_type                12000 non-null  str    
 4   construction_area_sqft       12000 non-null  int64  
 5   num_floors                   12000 non-null  int64  
 6   bedrooms                     12000 non-null  int64  
 7   bathrooms                

In [3]:
# Separate input features and target column

X = df.drop(columns=["project_id", "total_construction_cost_inr"])

y = df["total_construction_cost_inr"]

# Check the data
print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nInput Features:")
print(X.columns.tolist())

print("\nTarget:")
print(y.head())

Features shape: (12000, 13)
Target shape: (12000,)

Input Features:
['year', 'district', 'building_type', 'construction_area_sqft', 'num_floors', 'bedrooms', 'bathrooms', 'construction_quality', 'foundation_type', 'material_price_index', 'labor_rate_per_sqft', 'site_condition', 'project_duration_months']

Target:
0    4614000
1    7711000
2    3913000
3    3574000
4    2983000
Name: total_construction_cost_inr, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Check the size of each dataset
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining target:", y_train.shape)
print("Testing target:", y_test.shape)

Training data: (9600, 13)
Testing data: (2400, 13)

Training target: (9600,)
Testing target: (2400,)


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Find categorical columns
categorical_columns = X.select_dtypes(include="str").columns.tolist()

print("Categorical Columns:")
print(categorical_columns)

# Convert text categories into numerical values
preprocessor = ColumnTransformer(
    transformers=[
        ("category", OneHotEncoder(handle_unknown="ignore"), categorical_columns)
    ],
    remainder="passthrough"
)

# Apply encoding to training and testing data
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("\nTraining data after encoding:", X_train_encoded.shape)
print("Testing data after encoding:", X_test_encoded.shape)

Categorical Columns:
['district', 'building_type', 'construction_quality', 'foundation_type', 'site_condition']

Training data after encoding: (9600, 29)
Testing data after encoding: (2400, 29)


In [6]:
from sklearn.linear_model import LinearRegression

# Create the Linear Regression model
model = LinearRegression()

# Train the model
model.fit(X_train_encoded, y_train)

print("Linear Regression model trained successfully!")

Linear Regression model trained successfully!


In [7]:
# Make predictions using the trained model
y_pred = model.predict(X_test_encoded)

# Compare actual and predicted costs
comparison = pd.DataFrame({
    "Actual Cost": y_test.values,
    "Predicted Cost": y_pred.round(0)
})

print(comparison.head(10))

   Actual Cost  Predicted Cost
0      3380000       2689967.0
1      7493000       6931160.0
2      3959000       5746179.0
3      7243000       7114927.0
4      7325000       8225081.0
5      2847000       3145069.0
6      6370000       6886974.0
7      8553000       7964662.0
8      5647000       5419391.0
9      6793000       8986861.0


In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Calculate model performance
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Linear Regression Model Performance")
print("-----------------------------------")
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2 Score:", round(r2, 4))

Linear Regression Model Performance
-----------------------------------
MAE: 518555.06
RMSE: 781468.46
R2 Score: 0.9147


In [9]:
from sklearn.ensemble import RandomForestRegressor

# Create the Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

# Train the model
rf_model.fit(X_train_encoded, y_train)

print("Random Forest model trained successfully!")

Random Forest model trained successfully!


In [10]:
# Make predictions using Random Forest
rf_pred = rf_model.predict(X_test_encoded)

# Calculate performance
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_mse = mean_squared_error(y_test, rf_pred)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest Model Performance")
print("--------------------------------")
print("MAE:", round(rf_mae, 2))
print("RMSE:", round(rf_rmse, 2))
print("R2 Score:", round(rf_r2, 4))

Random Forest Model Performance
--------------------------------
MAE: 327204.59
RMSE: 491500.21
R2 Score: 0.9662


In [11]:
# Compare both machine learning models

results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [mae, rf_mae],
    "RMSE": [rmse, rf_rmse],
    "R2 Score": [r2, rf_r2]
})

print(results.round(2))

               Model        MAE       RMSE  R2 Score
0  Linear Regression  518555.06  781468.46      0.91
1      Random Forest  327204.59  491500.21      0.97


In [12]:
import joblib

# Save the trained model
joblib.dump(rf_model, "../models/construction_cost_model.pkl")

# Save the encoding process
joblib.dump(preprocessor, "../models/data_preprocessor.pkl")

print("Model saved successfully!")
print("Preprocessor saved successfully!")

Model saved successfully!
Preprocessor saved successfully!


In [13]:
# Create sample construction project details

sample_project = pd.DataFrame([{
    "year": 2026,
    "district": "Hyderabad",
    "building_type": "Independent House",
    "construction_area_sqft": 1500,
    "num_floors": 2,
    "bedrooms": 3,
    "bathrooms": 2,
    "construction_quality": "Standard",
    "foundation_type": "Shallow",
    "material_price_index": 150.0,
    "labor_rate_per_sqft": 750,
    "site_condition": "Normal",
    "project_duration_months": 10
}])

# Convert sample data using the saved preprocessor
sample_encoded = preprocessor.transform(sample_project)

# Predict construction cost
predicted_cost = rf_model.predict(sample_encoded)

print("Estimated Construction Cost: ₹", round(predicted_cost[0], 2))

Estimated Construction Cost: ₹ 4991380.0


In [14]:
import joblib

# Load saved model and preprocessor
loaded_model = joblib.load("../models/construction_cost_model.pkl")
loaded_preprocessor = joblib.load("../models/data_preprocessor.pkl")

# Prepare sample data
sample_encoded = loaded_preprocessor.transform(sample_project)

# Make prediction
prediction = loaded_model.predict(sample_encoded)

print("Estimated Construction Cost: ₹", round(prediction[0], 2))

Estimated Construction Cost: ₹ 4991380.0
